In [ ]:
#@title Install Java version 17 due to Neo4j requirements
!apt-get install openjdk-17-jre-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
!java -version

openjdk version "17.0.11" 2024-04-16
OpenJDK Runtime Environment (build 17.0.11+9-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.11+9-Ubuntu-122.04.1, mixed mode, sharing)


In [ ]:
#@title Installing Neo4J server on Google colab
!wget -q https://neo4j.com/artifact.php?name=neo4j-community-5.21.0-unix.tar.gz -O neo4j.tar.gz
# decompress and rename
!tar -xf neo4j.tar.gz  # or --strip-components=1
!mv neo4j-community-5.21.0 nj
# Allow apoc command
!wget -q https://github.com/neo4j/apoc/releases/download/5.21.0/apoc-5.21.0-core.jar -O nj/plugins/apoc-5.21.0-core.jar
!echo "dbms.security.procedures.unrestricted=apoc.*" >> nj/conf/neo4j.conf
!echo "dbms.security.procedures.allowlist=apoc.*" >> nj/conf/neo4j.conf
# disable password, and start server
!sed -i '/#dbms.security.auth_enabled/s/^#//g' nj/conf/neo4j.conf
# start Neo4j server
!nj/bin/neo4j start

Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:2827). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
#@title Install Mongo, Neo4j and Azure OpenAI python engines
%%capture
!pip install py2neo -q
!pip install llama-index -q
!pip install llama-index-llms-azure-openai -q
!pip install llama-index-embeddings-azure-openai -q
!pip install llama-index-vector-stores-neo4jvector -q
!pip install llama-index-graph-stores-neo4j -q
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from google.colab import userdata
from llama_index.core import Settings

llm = AzureOpenAI(
    engine="gpt-4o",
    deployment_name=userdata.get("AZURE_MODEL_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
    temperature=0.0001,
)

embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small",
    deployment_name=userdata.get("AZURE_EMBEDDING_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
)

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 500

In [ ]:
#@title Read a BILL PDF format
!gdown --id 1WLd-hmavGY6bSUR3gB3OO6aDhPaT_zwl

/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:132: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1WLd-hmavGY6bSUR3gB3OO6aDhPaT_zwl
To: /content/BILLS-118hr6363enr.pdf
100% 140k/140k [00:00<00:00, 21.2MB/s]


In [ ]:
#@title Load a Bill Document
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

parser = PDFReader()
file_extractor = {".pdf": parser}
documents = SimpleDirectoryReader(
    input_files=["./BILLS-118hr6363enr.pdf"], file_extractor=file_extractor
).load_data()

In [ ]:
documents[0]

Document(id_='75a01e68-a901-4cb8-9386-1fa6c254561c', embedding=None, metadata={'page_label': '1', 'file_name': 'BILLS-118hr6363enr.pdf', 'file_path': 'BILLS-118hr6363enr.pdf', 'file_type': 'application/pdf', 'file_size': 140196, 'creation_date': '2024-07-26', 'last_modified_date': '2024-07-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text='H. R. 6363 \nOne Hundred Eighteenth Congress \nof the \nUnited States of America \nAT THE FIRST SESSION \nBegun and held at the City of Washington on Tuesday, \nthe third day of January, two thousand and twenty-three \nAn Act \nMaking further continuing appropriations for fiscal year 2024, and for other pur-\nposes. \nBe it enacted by the Senate and House of Representatives of \nthe United States of America in Congr

In [ ]:
# count the number of documents
print(f"Number of documents: {len(documents)}")
from llama_index.core import Document
full_pdf = ''.join([doc.text for doc in documents])
documents = [Document(text=llm.complete(f"""
Be concise, paraphrase all US BILL text correctly, use bullet points, do not use any acronyms or acronyms like H.R.
please replace them by full name, the section, division and title must be redable for humans:

```{full_pdf}```


Paraphrase:""").text)]

Number of documents: 1


In [ ]:
documents[0].text

'- **House Resolution 6363**\n- **One Hundred Eighteenth Congress**\n- **United States of America**\n- **First Session**\n- **Started in Washington on Tuesday, January 3, 2023**\n\n**An Act**\n\n- **Purpose:** Providing additional continuing appropriations for fiscal year 2024, and for other purposes.\n\n**Enacted by the Senate and House of Representatives of the United States of America in Congress assembled:**\n\n**Section 1. Short Title**\n- This Act is called the "Further Continuing Appropriations and Other Extensions Act, 2024."\n\n**Section 2. Table of Contents**\n- The table of contents is as follows:\n  - Sec. 1. Short title.\n  - Sec. 2. Table of Contents.\n  - Sec. 3. References.\n\n**Division A—Further Continuing Appropriations Act, 2024**\n\n**Division B—Other Matters**\n- Title I—Extensions and Other Matters\n- Title II—Health and Human Services\n- Title III—Miscellaneous Extensions\n- Title IV—Budgetary Effects\n\n**Section 3. References**\n- Any reference to "this Act" i

In [ ]:
# Enable async python function in the Jupyer notebook
import nest_asyncio
nest_asyncio.apply()

In [ ]:
!nj/bin/neo4j restart

Stopping Neo4j............ stopped.
Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:11186). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
#@title Create a pointer to Neo4j Property Graph Index
from llama_index.graph_stores.neo4j import Neo4jPGStore

username=""
password=""
url="bolt://localhost:7687"

# Neo4j Community version only support the default database
database="neo4j"

graph_store = Neo4jPGStore(
    username=username,
    password=password,
    url=url,
    # database=database
)

In [ ]:
from py2neo import Graph
url = "bolt://localhost:7687"
graph = Graph(url)

In [ ]:
query = """
MATCH (m)-[t]->(r)
DETACH DELETE m,t,r
"""
graph.run(query).to_table()
query = """
MATCH (m)
DETACH DELETE m
"""
graph.run(query).to_table()

In [ ]:
#@title Entities and Relations specifications
from typing import Literal

entities = Literal[
    "BILL",
    "SECTION_DIVISION_TITLE",
    "AMENDMENT",
    "FUND",
    "PROGRAM",
    "ACT",
    "DEPARTMENT",
    "AGENCY"
]

relations = Literal[
    "HAS_SECTION",
    "HAS_DIVISION",
    "HAS_TITLE",
    "AMENDS",
    "FUNDS",
    "IS_PART_OF",
    "ADMINISTERED_BY",
    "EXTENDS"
]

validation_schema = {
    "Bill": ["HAS_SECTION", "HAS_DIVISION", "HAS_TITLE"],
    "SECTION_DIVISION_TITLE": ["AMENDS", "FUNDS", "EXTENDS"],
    "Appropriation": ["FUNDS"],
    "Amendment": ["AMENDS"],
    "Fund": ["FUNDS"],
    "Program": ["IS_PART_OF", "ADMINISTERED_BY"],
    "Act": ["EXTENDS"],
    "Department": ["ADMINISTERED_BY"],
    "Agency": ["ADMINISTERED_BY"]
}

In [ ]:
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from llama_index.core import PropertyGraphIndex
prompt =  """Give the following text, extract the knowledge graph according to the provided schema(All node names are in lower cases):
Try to limit to the output {max_triplets_per_chunk} extracted paths.
-------
{text}
-------"""
kg_extractor = SchemaLLMPathExtractor(
    extract_prompt=prompt,
    llm=llm,
    possible_entities=entities,
    possible_relations=relations,
    kg_validation_schema=validation_schema,
    strict=True, # # if false, allows values outside of spec
    # kg_schema_cls=kg_schema_cls,
    # num_workers=4,
    # max_triplets_per_chunk=10,
)

# Use the LLM and embed models to create the graph
index = PropertyGraphIndex.from_documents(
    documents[::], #  documents[:5], documents[::2](mitad), documents[:10]+documents[-1]
    kg_extractors=[kg_extractor],
    llm=llm,
    embed_model=embed_model,
    property_graph_store=graph_store,
    show_progress=True,
)

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings: 100%|██████████| 4/4 [00:00<00:00,  7.55it/s]


In [ ]:
query = """
MATCH (n)
RETURN DISTINCT labels(n) AS entity_classes
"""

graph.run(query).to_table()

entity_classes
['Chunk']
"['__Entity__', 'BILL']"
"['__Entity__', 'SECTION_DIVISION_TITLE']"
"['__Entity__', 'ACT']"
"['__Entity__', 'PROGRAM']"
"['__Entity__', 'FUND']"


In [ ]:
query = """
MATCH ()-[r]->()
RETURN DISTINCT type(r) AS relationship_types
"""

graph.run(query).to_table()

relationship_types
MENTIONS
HAS_SECTION
HAS_DIVISION
EXTENDS


In [ ]:
query = """
MATCH ()-[r]->()
RETURN type(r) AS relationship_type, COUNT(*) AS count
ORDER BY count DESC
"""

graph.run(query).to_table()

relationship_type,count
MENTIONS,25
EXTENDS,13
HAS_SECTION,3
HAS_DIVISION,2


In [ ]:
query = """
MATCH (n)
WITH DISTINCT n, apoc.map.removeKeys(properties(n), ['embedding','triplet_source_id','id','text']) AS data
RETURN data, labels(n) AS entity_classes
"""
graph.run(query).to_table()

data,entity_classes
"{_node_type: 'TextNode', _node_content: '{""id_"": ""5e29a1e9-adb1-4beb-be11-91f229099cb9"", ""embedding"": null, ""metadata"": {}, ""excluded_embed_metadata_keys"": [], ""excluded_llm_metadata_keys"": [], ""relationships"": {""1"": {""node_id"": ""09ecde5e-c940-4778-b5b9-01d694b53a18"", ""node_type"": ""4"", ""metadata"": {}, ""hash"": ""78786ad384e85728b8430802d8c6356faec6ab7424453a5ad594cc4209a62755"", ""class_name"": ""RelatedNodeInfo""}, ""3"": {""node_id"": ""43852d39-00d6-4e40-9a13-d6ef29ece551"", ""node_type"": ""1"", ""metadata"": {}, ""hash"": ""3f4e73d7c47f2f1b572397d4aab47a587ad2774cdb3163c7e358f17865a25982"", ""class_name"": ""RelatedNodeInfo""}}, ""text"": """", ""mimetype"": ""text/plain"", ""start_char_idx"": 0, ""end_char_idx"": 2000, ""text_template"": ""{metadata_str}\\n\\n{content}"", ""metadata_template"": ""{key}: {value}"", ""metadata_seperator"": ""\\n"", ""class_name"": ""TextNode""}', document_id: '09ecde5e-c940-4778-b5b9-01d694b53a18', doc_id: '09ecde5e-c940-4778-b5b9-01d694b53a18', ref_doc_id: '09ecde5e-c940-4778-b5b9-01d694b53a18'}",['Chunk']
"{_node_type: 'TextNode', _node_content: '{""id_"": ""43852d39-00d6-4e40-9a13-d6ef29ece551"", ""embedding"": null, ""metadata"": {}, ""excluded_embed_metadata_keys"": [], ""excluded_llm_metadata_keys"": [], ""relationships"": {""1"": {""node_id"": ""09ecde5e-c940-4778-b5b9-01d694b53a18"", ""node_type"": ""4"", ""metadata"": {}, ""hash"": ""78786ad384e85728b8430802d8c6356faec6ab7424453a5ad594cc4209a62755"", ""class_name"": ""RelatedNodeInfo""}, ""2"": {""node_id"": ""5e29a1e9-adb1-4beb-be11-91f229099cb9"", ""node_type"": ""1"", ""metadata"": {}, ""hash"": ""cbd934326ff4c03fc8238cae4330684f664b09f98f6775fd8421d2f275d622b1"", ""class_name"": ""RelatedNodeInfo""}, ""3"": {""node_id"": ""72481a0a-d875-4ec1-b2e6-3821455eab27"", ""node_type"": ""1"", ""metadata"": {}, ""hash"": ""12ab7a2bd95ddaf13aed6ae7d596b39aad7dbc1ac99167149b7f762bf70f0d42"", ""class_name"": ""RelatedNodeInfo""}}, ""text"": """", ""mimetype"": ""text/plain"", ""start_char_idx"": 1120, ""end_char_idx"": 3217, ""text_template"": ""{metadata_str}\\n\\n{content}"", ""metadata_template"": ""{key}: {value}"", ""metadata_seperator"": ""\\n"", ""class_name"": ""TextNode""}', document_id: '09ecde5e-c940-4778-b5b9-01d694b53a18', doc_id: '09ecde5e-c940-4778-b5b9-01d694b53a18', ref_doc_id: '09ecde5e-c940-4778-b5b9-01d694b53a18'}",['Chunk']
"{_node_type: 'TextNode', _node_content: '{""id_"": ""72481a0a-d875-4ec1-b2e6-3821455eab27"", ""embedding"": null, ""metadata"": {}, ""excluded_embed_metadata_keys"": [], ""excluded_llm_metadata_keys"": [], ""relationships"": {""1"": {""node_id"": ""09ecde5e-c940-4778-b5b9-01d694b53a18"", ""node_type"": ""4"", ""metadata"": {}, ""hash"": ""78786ad384e85728b8430802d8c6356faec6ab7424453a5ad594cc4209a62755"", ""class_name"": ""RelatedNodeInfo""}, ""2"": {""node_id"": ""43852d39-00d6-4e40-9a13-d6ef29ece551"", ""node_type"": ""1"", ""metadata"": {}, ""hash"": ""3f4e73d7c47f2f1b572397d4aab47a587ad2774cdb3163c7e358f17865a25982"", ""class_name"": ""RelatedNodeInfo""}}, ""text"": """", ""mimetype"": ""text/plain"", ""start_char_idx"": 2454, ""end_char_idx"": 4148, ""text_template"": ""{metadata_str}\\n\\n{content}"", ""metadata_template"": ""{key}: {value}"", ""metadata_seperator"": ""\\n"", ""class_name"": ""TextNode""}', document_id: '09ecde5e-c940-4778-b5b9-01d694b53a18', doc_id: '09ecde5e-c940-4778-b5b9-01d694b53a18', ref_doc_id: '09ecde5e-c940-4778-b5b9-01d694b53a18'}",['Chunk']
{name: 'house resolution 6363'},"['__Entity__', 'BILL']"
{name: 'section 1'},"['__Entity__', 'SECTION_DIVISION_TITLE']"
{name: 'section 2'},"['__Entity__', 'SECTION_DIVISION_TITLE']"
{name: 'section 3'},"['__Entity__', 'SECTION_DIVISION_TITLE']"
{name: 'division a'},"['__Entity__', 'SECTION_DIVISION_TITLE']"
{name: 'division b'},"['__Entity__', 'SECTION_DIVISION_TITLE']"
{name: 'section 101'},"['__Entity__', 'SECTION_DIVISION_TITLE']"


In [ ]:
query = """
MATCH (n:PROGRAM)
WITH apoc.map.removeKeys(properties(n), ['embedding','triplet_source_id','id','text']) AS data
RETURN data
"""
graph.run(query).to_table()

In [ ]:
query = """
MATCH (n:SECTION)
WITH apoc.map.removeKeys(properties(n), ['embedding','triplet_source_id','id']) AS data
RETURN data
LIMIT 1
"""

graph.run(query).to_table()

In [ ]:
query = """
MATCH (n:TITLE)
WITH apoc.map.removeKey(properties(n), 'embedding') AS titles
RETURN titles
"""
graph.run(query).to_table()

titles
"{id: 'further continuing appropriations act, 2024', name: 'further continuing appropriations act, 2024', triplet_source_id: '577edf53-8227-4689-9685-228b0bbf37b4'}"
"{id: 'title i', name: 'title i', triplet_source_id: '577edf53-8227-4689-9685-228b0bbf37b4'}"
"{id: 'title ii', name: 'title ii', triplet_source_id: '577edf53-8227-4689-9685-228b0bbf37b4'}"
"{id: 'title iii', name: 'title iii', triplet_source_id: '577edf53-8227-4689-9685-228b0bbf37b4'}"
"{id: 'title iv', name: 'title iv', triplet_source_id: '577edf53-8227-4689-9685-228b0bbf37b4'}"


In [ ]:
query = """
MATCH (n:ACT)
WITH apoc.map.removeKey(properties(n), 'embedding') AS Act
RETURN Act
"""
graph.run(query).to_table()

In [ ]:
query = """
MATCH (n)-[r:AMENDS]->(m)
WITH apoc.map.removeKey(properties(n), 'embedding') AS N,
apoc.map.removeKey(properties(m), 'embedding') AS M,
apoc.map.removeKey(properties(r), 'embedding') AS R
RETURN N.name,R,M.name
"""
graph.run(query).to_table()

In [ ]:
#@title Install libs to display Neo4j Graph
!pip install yfiles_jupyter_graphs_for_neo4j -q
from google.colab import output
output.enable_custom_widget_manager()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.4 MB/s eta 0:00:00


In [ ]:
#@title Load Visualization Component
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget
from neo4j import GraphDatabase

driver = GraphDatabase.driver(uri = url,)

g = Neo4jGraphWidget(driver)

In [ ]:
query = """
MATCH (c:SECTION)
RETURN c
LIMIT 15
"""

g.show_cypher(query)

GraphWidget(layout=Layout(height='650px', width='100%'))

In [ ]:
query = """
MATCH (c:TITLE)
RETURN c
LIMIT 15
"""

g.show_cypher(query)

GraphWidget(layout=Layout(height='620px', width='100%'))

In [ ]:
query = """
MATCH (n)-[r]->(m)
WITH apoc.map.removeKey(properties(n), 'embedding') AS N,
apoc.map.removeKey(properties(m), 'embedding') AS M,
apoc.map.removeKey(properties(r), 'embedding') AS R
RETURN N,R,M
"""

g.show_cypher(query)

GraphWidget(layout=Layout(height='500px', width='100%'))

In [ ]:
from pydantic import BaseModel
from typing import Optional, List

# Must use names for Neo4J identification
class Entities(BaseModel):
    """List of named entities in the text such as names of candidates, professions, organizations, skills, references and job descriptions"""
    names: Optional[List[str]]

prompt_template_entities = """
Think carefully the following text and Extract all named entities such as names of candidates, professions, organizations, hard_skills, soft_skills, references, educations, projects, courses and job descriptions
from the following text:
{text}
"""

In [ ]:
from typing import Any, Optional

from llama_index.core.embeddings import BaseEmbedding
from llama_index.core.retrievers import CustomPGRetriever, VectorContextRetriever
from llama_index.core.vector_stores.types import VectorStore
from llama_index.program.openai import OpenAIPydanticProgram


class CWSRetriever(CustomPGRetriever):
    """Custom retriever with cohere reranking."""

    def init(
        self,
        embed_model: Optional[BaseEmbedding] = None,
        vector_store: Optional[VectorStore] = None,
        similarity_top_k: int = 5,
        path_depth: int = 1,
        include_text: bool = True,
        **kwargs: Any,
    ) -> None:
        """Uses any kwargs passed in from class constructor."""
        self.entity_extraction = OpenAIPydanticProgram.from_defaults(
            output_cls=Entities, prompt_template_str=prompt_template_entities
        )
        self.vector_retriever = VectorContextRetriever(
            self.graph_store,
            include_text=self.include_text,
            embed_model=embed_model,
            similarity_top_k=similarity_top_k,
            path_depth=path_depth,
        )

    def custom_retrieve(self, query_str: str) -> str:
        """Define custom retriever with reranking.

        Could return `str`, `TextNode`, `NodeWithScore`, or a list of those.
        """
        temp_result = self.entity_extraction(text=query_str)
        entities = temp_result.names
        result_nodes = []
        if entities:
            print(f"Detected entities: {entities}")
            for entity in entities:
                result_nodes.extend(self.vector_retriever.retrieve(entity))
        # else:
        result_nodes.extend(self.vector_retriever.retrieve(query_str))
        final_text = "\n\n".join(
            [n.get_content(metadata_mode="llm") for n in result_nodes]
        )
        return final_text

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

custom_sub_retriever = CWSRetriever(
    index.property_graph_store,
    include_text=True,
    vector_store=index.vector_store,
    embed_model=embed_model
)

query_engine = RetrieverQueryEngine.from_args(
    index.as_retriever(sub_retrievers=[custom_sub_retriever]), llm=llm
)

In [ ]:
response = query_engine.query("Give me skills related to Full Stack Java Developer")
print(str(response))

Skills related to a Full Stack Java Developer include Java, Angular, ReactJS, CSS, JavaScript, HTML, Java Enterprise Edition (J2EE), XML, Web Services, Spring Framework, JMS, SOAP, REST web services, and AWS Cloud.


In [ ]:
response = query_engine.query("list all hard kills")
print(str(response))

Here is a list of all hard skills:

1. Windows XP/Windows 7/Linux
2. HTML
3. C#
4. CSS
5. .NET
6. SQL SERVER2008
7. Fluid mechanics
8. Advanced Manufacturing Technology
9. Machine Element Designing
10. Engine components, pumps and fuel
11. Pressure vessel design knowledge
12. C
13. C++
14. JAVA
15. DBMS
16. Knowledge in Heat exchanger, automobile
17. Autocad
18. MS Word
19. Work on Excel Sheet
20. Good command on welding defect and its impact
21. Knowledge about welding (ARC, TIG, MIG, SAW, GAS)
22. Cleanliness of plant & machinery, inspection measuring & test equipment
23. Pipe fitting & welding
24. Major focus to zero seven types of losses from processes
25. Java
26. Python
27. NoSQL
28. SAP (System Application & Program)
29. Oracle
30. Read soft
31. Birch Street (BST)
32. MS Office
33. Data Quant
34. Support Origin
35. Excel (MIS)
36. Microsoft office suite
37. Tally 9.0
38. Proficient in MS Office
39. AutoCAD/2004/2007/13, 14 & 15
40. Small world 2.1
41. Ground V10 & CMD software
4